<a href="https://colab.research.google.com/github/Chimix001/Gemmacode/blob/main/GemmaCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate  qdrant-client sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 7.0 MB/s eta 0:00:00


In [ ]:
!pip install -q bitsandbytes>=0.46.1

In [ ]:
import torch
from qdrant_client import models
from qdrant_client import QdrantClient
from transformers import AutoProcessor, Gemma4ForConditionalGeneration
import base64
from io import BytesIO
from tqdm import tqdm


LOADING THE GEMMA4 MODEL FROM HUGGING FACE:

In [ ]:
from transformers import (
    AutoProcessor,
    Gemma4ForConditionalGeneration,
    BitsAndBytesConfig,
)

model_id =  "google/gemma-4-E4B-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(model_id)

model = Gemma4ForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

CONVERSATION MEMORY

In [ ]:
messages = [
    {
        "role": "user",
        "content": [{"type": "text", "text": "Write a Python function that implements binary search."}],
    }
]

input_ids = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(
    input_ids=input_ids,
    max_new_tokens=4092,
    temperature=0.2,
)

print(processor.decode(outputs[0], skip_special_tokens=True))

user
Write a Python function that implements binary search.
model
Here is a Python function that implements binary search. I'll provide two common variations: one for finding an element's presence (boolean result) and another for finding the index (returning the index or -1).

### 1. Binary Search to Find Presence (Boolean Result)

This version is useful if you only need to know *if* the item exists in the sorted list.

```python
def binary_search_presence(arr: list, target: int) -> bool:
    """
    Searches for a target value in a sorted list using binary search.

    Args:
        arr: The sorted list of numbers to search within.
        target: The value to search for.

    Returns:
        True if the target is found, False otherwise.
    """
    low = 0
    high = len(arr) - 1

    while low <= high:
        # Calculate the middle index. Using (low + high) // 2 is standard,
        # but low + (high - low) // 2 helps prevent potential integer overflow 
        # in languages with

In [ ]:
finish_reason = "complete" if outputs[0][-1] == processor.tokenizer.eos_token_id else "truncated"

GENERATION CODE

In [ ]:
def generate_code(prompt):
    messages = [
        {
            "role": "user",
            "content": [{"type": "text", "text": prompt}],
        }
    ]

    input_ids = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=4096,
        temperature=0.2,
        do_sample=True,
    )

    response = processor.decode(outputs[0], skip_special_tokens=True)

    # Remove the echoed prompt if present
    if "model" in response:
        response = response.split("model", 1)[1].strip()

    return response

SYSTEM PROMPT

In [ ]:
# system prompt

SYSTEM_PROMPT = """
You are GemmaCode, an expert AI software engineer.

Rules:

1. Write clean, production-ready code.
2. Explain your reasoning briefly.
3. Fix bugs.
4. Refactor code when requested.
5. Build complete applications.
6. Follow software engineering best practices.
7. Generate complete files whenever possible.
8. If requirements are unclear, ask concise clarifying questions.
9. Always format code using Markdown code blocks.
10. When creating projects, generate the folder structure first.
"""

# Conversation Memory

conversation_history = [
    {
        "role": "system",
        "content": [
            {
                "type": "text",
                "text": SYSTEM_PROMPT
            }
        ]
    }
]

# Keep only the latest conversations
MAX_HISTORY = 10


# RAG Placeholder

def retrieve_context(query):
    """
    Later this will search Qdrant / FAISS / ChromaDB.

    For now it returns nothing.
    """
    return ""


# Generate Response

def generate_code(prompt):

    global conversation_history

    # Retrieve documentation (future RAG)

    context = retrieve_context(prompt)

    if context:

        prompt = f"""
Reference Documentation

{context}

User Request

{prompt}
"""

    # Save user message

    conversation_history.append(
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }
    )

    # Limit conversation length

    if len(conversation_history) > MAX_HISTORY:

        conversation_history = (
            conversation_history[:1] +
            conversation_history[-(MAX_HISTORY-1):]
        )

    # Convert to model input

    input_ids = processor.apply_chat_template(
        conversation_history,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    # Generate response

    with torch.inference_mode():

        outputs = model.generate(
            input_ids=input_ids,
            max_new_tokens=2048,
            temperature=0.2,
            do_sample=True,
            top_p=0.95,
            repetition_penalty=1.05,
        )

    # Decode only newly generated tokens

    generated_tokens = outputs[0][input_ids.shape[-1]:]

    response = processor.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    # Save assistant response

    conversation_history.append(
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": response
                }
            ]
        }
    )

    return response


# Clear Conversation

def clear_chat():

    global conversation_history

    conversation_history = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": SYSTEM_PROMPT
                }
            ]
        }
    ]

    print("Conversation cleared.")

In [ ]:
# GemmaCode Task Router

TASK_PROMPTS = {

    "generate": """
You are GemmaCode.

Generate complete, production-ready code.

Always:
- Produce complete files.
- Use best practices.
- Add useful comments only.
- Explain briefly before the code.
""",

    "debug": """
You are GemmaCode.

You are debugging code.

Always:
- Find every bug.
- Explain each bug.
- Show the corrected code.
- Explain why the fix works.
""",

    "review": """
You are GemmaCode.

Review the user's code.

Look for:

- Bugs
- Security issues
- Performance problems
- Readability
- Maintainability
- Best practices

Provide suggestions before rewriting.
""",

    "refactor": """
You are GemmaCode.

Refactor the user's code.

Goals:

- Cleaner structure
- Better variable names
- Better performance
- Simpler logic
- Preserve functionality
""",

    "explain": """
You are GemmaCode.

Explain the code clearly.

Teach like a senior software engineer mentoring a junior developer.

Break the explanation into sections.
"""
}


# Task Dispatcher

def run_task(task, prompt):

    if task not in TASK_PROMPTS:
        task = "generate"

    full_prompt = f"""
{TASK_PROMPTS[task]}

User Request:

{prompt}
"""

    return generate_code(full_prompt)

In [11]:
SAVE_PATH = "/content/drive/MyDrive/GemmaCode/model"

model.save_pretrained(SAVE_PATH)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
processor.save_pretrained(SAVE_PATH)

['/content/drive/MyDrive/GemmaCode/model/processor_config.json']

In [ ]:
import os

SAVE_PATH = "/content/drive/MyDrive/GemmaCode/model"

print(os.path.exists(SAVE_PATH))
print(os.listdir(SAVE_PATH))

True
['processor_config.json', 'model.safetensors', 'chat_template.jinja', 'tokenizer_config.json', 'config.json', 'generation_config.json', 'tokenizer.json']


In [ ]:
import os

print(os.path.abspath(SAVE_PATH))

/content/drive/MyDrive/GemmaCode/model
